# Hibiki-Sw Stage 4: Fine-tuning + Evaluation

**GPU Budget: ~3 hours** (4k steps fine-tuning + evaluation, 2x T4)

Fine-tune on curated aligned data with classifier-free guidance,
then evaluate on FLEURS test set.

**Prerequisites:**
- Stage 3 checkpoint
- Fine-tuning manifest

In [ ]:
!pip install -q transformers accelerate sentencepiece pyyaml tensorboard \
    faster-whisper sacrebleu soundfile scipy datasets

In [ ]:
import os
import torch

REPO_DIR = "/kaggle/input/hibiki-sw-code/hibiki-sw"
DATA_DIR = "/kaggle/input/hibiki-sw-data"
OUTPUT_DIR = "/kaggle/working/stage4"
EVAL_DIR = "/kaggle/working/eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

STAGE3_CKPT = "/kaggle/input/hibiki-sw-stage3/checkpoint_final.pt"
MANIFEST = f"{DATA_DIR}/manifests/fleurs_en2sw_train.tsv"
TOKENIZER = f"{DATA_DIR}/tokenizer/sp_ensw_32k.model"

In [ ]:
!cp -r {REPO_DIR}/* /kaggle/working/hibiki-sw/

## Stage 4: Fine-tuning (~2 hours)

In [ ]:
%%time
!cd /kaggle/working/hibiki-sw && torchrun \
    --nproc_per_node=2 \
    --master_port=29500 \
    training/train_finetune.py \
    --config configs/model_100m.yaml \
    --manifest {MANIFEST} \
    --stage3_ckpt {STAGE3_CKPT} \
    --output_dir {OUTPUT_DIR}

## Evaluation on FLEURS (~1 hour)

In [ ]:
%%time
FINAL_CKPT = f"{OUTPUT_DIR}/checkpoint_final.pt"

!cd /kaggle/working/hibiki-sw && python evaluation/evaluate.py \
    --checkpoint {FINAL_CKPT} \
    --config configs/model_100m.yaml \
    --eval_set fleurs \
    --output_dir {EVAL_DIR} \
    --tokenizer {TOKENIZER} \
    --max_samples 100 \
    --direction en2sw

In [ ]:
# Display results
import json

with open(f"{EVAL_DIR}/metrics.json") as f:
    metrics = json.load(f)

print("=" * 50)
print("Hibiki-Sw Evaluation Results")
print("=" * 50)
print(f"Samples evaluated: {metrics.get('num_samples', 'N/A')}")
print()

if 'asr_bleu' in metrics:
    print(f"ASR-BLEU:   {metrics['asr_bleu']['bleu']:.2f}")
if 'text_bleu' in metrics:
    print(f"Text BLEU:  {metrics['text_bleu']['bleu']:.2f}")
if 'latency' in metrics:
    lat = metrics['latency']
    print(f"Avg RTF:    {lat['avg_rtf']:.2f}x")
    print(f"Avg Time:   {lat['avg_total_time']:.2f}s")
print("=" * 50)

## Quick Demo: Translate a sample

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/hibiki-sw")
import yaml
import IPython.display as ipd
from inference.translate import load_model, load_audio, translate, decode_text
from model.codec import MimiCodec

with open("/kaggle/working/hibiki-sw/configs/model_100m.yaml") as f:
    config = yaml.safe_load(f)

model = load_model(config, FINAL_CKPT, "cuda")
codec = MimiCodec(num_codebooks=8, device="cuda")

In [ ]:
# Load a FLEURS test sample
from datasets import load_dataset
import torch, torchaudio

ds = load_dataset("google/fleurs", "en_us", split="test")
sample = ds[0]

audio = torch.tensor(sample["audio"]["array"], dtype=torch.float32).unsqueeze(0)
sr = sample["audio"]["sampling_rate"]
if sr != 24000:
    audio = torchaudio.transforms.Resample(sr, 24000)(audio)

print(f"Source text: {sample['transcription']}")
print("\nSource audio:")
ipd.display(ipd.Audio(audio.squeeze().numpy(), rate=24000))

In [ ]:
# Translate!
result = translate(model, codec, audio, device="cuda")

text = decode_text(result["translated_text_ids"], TOKENIZER)
print(f"\nTranslated text (inner monologue): {text}")
print("\nTranslated audio:")
ipd.display(ipd.Audio(result["translated_audio"].squeeze().numpy(), rate=24000))